# Meta x PyTorch x HuggingFace Hackathon (Round 2)

## Fine-Tuning Llama-3 (8B) using Unsloth & OpenEnv Invotex 🚀

This notebook provides the minimal training pipeline to connect your live `openenv_invotex` HuggingFace Space to an Unsloth/TRL SFT Training loop. Our environment perfectly models dynamic **Schema Drift** for True Agent Evaluation.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install 'openenv-core[core]>=0.2.1'

### 1. Load Unsloth Model & tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # OpenEnv traces are relatively long due to system prompts

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",    
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### 2. Format Environmental Trajectories for Llama-3

In [ ]:
from datasets import Dataset
import json

# In a full pipeline, we would export traces from 'openenv_invotex'. 
# Here we create a dummy high-quality trajectory targeting our Schema Drift (403 Error)
data = [
    {
        "messages": [
            {"role": "system", "content": "You are an OpenEnv Customer Support Agent..."},
            {"role": "user", "content": "Here is your task: Process a refund for ticket T-002."},
            {"role": "assistant", "content": "{\"tool_name\": \"process_refund\", \"tool_args\": {\"ticket_id\": \"T-002\", \"amount\": 50.0, \"reason\": \"broken item\"}}"},
            # Dynamic Schema Drift Error here!
            {"role": "user", "content": "API ERROR: 403 Forbidden - Schema Validation Failed. The /v2/refunds endpoint now requires an 'authorization_code'."},
            # Ideal Agent Recovery
            {"role": "assistant", "content": "{\"tool_name\": \"lookup_customer\", \"tool_args\": {\"customer_id\": \"C-002\"}}"},
        ]
    }
]

def formatting_prompts_func(example):
    return { "text" : tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False) }

dataset = Dataset.from_list(data)
dataset = dataset.map(formatting_prompts_func, batched = False)

### 3. Initialize SFTTrainer & Train

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()